In [ ]:
library('dplyr')
library('ggplot2')
library('forcats')

options(repr.plot.width = 5.5, repr.plot.height = 6.75)

We are going to reproduce this figure  from a [FiveThirtyEight](https://en.wikipedia.org/wiki/FiveThirtyEight) article.

<img src="https://ia802809.us.archive.org/9/items/fivethirtyeight-image-0587fca265a9/0587fca265a99bbe82a37c96b68f952be7cf2462.png" style="height: 600px"/>

## Data prep

All the wrangling happens once, here:

- `time`: parse `"2H 7M 57S"` into a Duration (seconds under the hood).
- `MF`: a factor whose level order sets the panel order (Women's on top).
- `country_group`: keep four countries and lump the rest into `"Other"` with `forcats::fct_other()`. This is what we will color by.

In [ ]:
mf_levels      = c("Women's", "Men's")
keep_countries = c('United States', 'Kenya', 'Ethiopia', 'Japan')

marathon = read.csv('https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/refs/heads/main/data/marathon.csv') |>
    mutate(
        time          = lubridate::as.duration(time),
        MF            = factor(MF, levels = mf_levels),
        country_group = fct_other(country, keep = keep_countries, other_level = 'Other')
    )

marathon |> glimpse()

## Step 0: the basic chart

In [ ]:
p0 = ggplot(marathon, aes(x = year, y = time, color = country_group)) + 
    geom_point(size = 2.5, alpha = 0.7) +
    facet_wrap(~MF, scales = 'free', ncol = 1)

p0

## Step 1: theme

FiveThirtyEight uses one light gray background for the whole figure, slightly darker gray grid lines, no minor grid, no ticks, and no legend (countries get labeled on the plot instead).

In [ ]:
bg = '#f0f0f0'
gc = '#dddddd'

p1 = p0 +
    theme(
        plot.background  = element_rect(fill = bg, color = NA),
        panel.background = element_rect(fill = bg, color = NA),
        panel.grid.major = element_line(color = gc, linewidth = 0.6),
        panel.grid.minor = element_blank(),
        axis.ticks       = element_blank(),
        strip.background = element_blank(),
        strip.text       = element_blank(),
        legend.position  = 'none',
        plot.margin      = margin(15, 25, 10, 15)
    )

p1

## Step 2: text

Gray axis numbers, no x title, and a bold y title.

In [ ]:
panel_labels = data.frame(
    MF    = factor(mf_levels, levels = mf_levels),
    label = toupper(mf_levels),
    year  = 2013,
    time  = Inf
)

panel_labels |> glimpse()

In [ ]:
p2 = p1 +
    geom_text(data = panel_labels, aes(label = label),
              hjust = 1, vjust = 1.5, fontface = 'bold', size = 5, color = 'black') +
    labs(x = NULL, y = 'Winning time') +
    theme(
        axis.text    = element_text(color = '#999999', size = 11),
        axis.title.y = element_text(face = 'bold', size = 13)
    )

p2

## Step 3: scales

- y labels as h:mm, breaks every 30 minutes.
- x labels as 1900, '10, '20, ... with '18 as the last break.
- Per-panel axis ranges via `geom_blank()`.

`panel_limits` holds the values that set the range of each facet panel:

- **y:** 2:00 to 3:30 for women, 2:00 to 3:00 for men (as in the original).
- **x:** left edge 1966 / 1895; right edge chosen so that 2018 sits 96% of the way across *both* panels. That is what puts the 2018 marker (step 6) at the same spot in each panel. ggplot's default 5% x padding would undo this, so it is turned off with `expand`.

In [ ]:
hmm        = function(sec) sprintf('%d:%02d', sec %/% 3600, round((sec %% 3600) / 60))
short_year = function(y)   ifelse(y %% 100 == 0, y, sprintf("'%02d", y %% 100))
same_spot  = function(start, at = 2018, frac = 0.96) start + (at - start) / frac

year_breaks = c(seq(1900, 2010, by = 10), 2018)

panel_limits = tribble(
    ~MF,       ~year,            ~time,
    "Women's", 1966,             2,
    "Women's", same_spot(1966),  3.5,
    "Men's",   1895,             2,
    "Men's",   same_spot(1895),  3
) |>
    mutate(MF = factor(MF, levels = mf_levels), time = time * 3600)

panel_limits |> glimpse()

In [ ]:
p3 = p2 +
    geom_blank(data = panel_limits, aes(color = NULL)) +
    scale_y_continuous(breaks = seq(0, 4 * 3600, by = 1800), labels = hmm) +
    scale_x_continuous(breaks = year_breaks, labels = short_year, expand = expansion(mult = 0))

p3

## Step 4: color

One FiveThirtyEight color per country group, sampled from the original image. `"Other"` is light gray so it fades back.

You can use the color dripper tool in Chrome devtools to find these hex codes.

In [ ]:
country_colors = c(
    'United States' = '#3fc1c9',
    'Kenya'         = '#9c4e97',
    'Ethiopia'      = '#48a949',
    'Japan'         = '#f05b4f',
    'Other'         = '#c8c8c8'
)

p4 = p3 + scale_color_manual(values = country_colors)

p4

## Step 5: country labels

One row per label: which panel, the text, and the (year, time) position, picked by eye from the original. The `country_group` column lets the same color scale as the points paint each label, so the two always match.

In [ ]:
country_labels = tribble(
    ~MF,       ~country_group,  ~label,            ~year, ~time,
    "Women's", 'United States', 'United\nStates',   1976,  3 +  5/60,
    "Women's", 'Ethiopia',      'Ethiopia',         1996,  2 + 33/60,
    "Women's", 'Kenya',         'Kenya',            2007,  2 + 15/60,
    "Men's",   'United States', 'United\nStates',   1912,  2 + 42/60,
    "Men's",   'Japan',         'Japan',            1952,  2 + 10/60,
    "Men's",   'Kenya',         'Kenya',            1994,  2 +  2/60
) |>
    mutate(
        MF   = factor(MF, levels = mf_levels),
        time = time * 3600   # hours -> seconds, to match the data
    )

country_labels |> glimpse()

In [ ]:
p5 = p4 +
    geom_text(data = country_labels, aes(label = label),
              hjust = 0, fontface = 'bold', size = 4.5, lineheight = 0.9)

p5

## Step 6: highlight 2018 marathon

Three pieces:

1. A dotted vertical line at 2018.
2. The two 2018 winners redrawn as open circles on top of the others.
3. Their names plus a curved arrow, positioned from a small data frame just like the country labels. Text sits at (`year`, `time`); the arrow runs from (`x`, `y`) to (`xend`, `yend`). All times are in hours and converted to seconds.

In [ ]:
winners_2018 = marathon |> filter(year == 2018)

winner_notes = tribble(
    ~MF,       ~label,             ~year, ~time,       ~x,   ~y,          ~xend,  ~yend,
    "Women's", 'Desiree\nLinden',   2013,  2 + 56/60,   2013, 2 + 50/60,   2016.8, 2 + 41/60,
    "Men's",   'Yuki\nKawauchi',    2011,  2 + 23/60,   2011, 2 + 20/60,   2016,   2 + 16.5/60
) |>
    mutate(
        MF = factor(MF, levels = mf_levels),
        across(c(time, y, yend), \(h) h * 3600)
    )

p6 = p5 +
    geom_vline(xintercept = 2018, linetype = 'dotted', color = '#999999', linewidth = 0.8) +
    geom_point(data = winners_2018, shape = 21, fill = bg, size = 3, stroke = 1.3) +
    geom_text(data = winner_notes, aes(label = label),
              hjust = 1, color = '#333333', size = 4, lineheight = 0.9) +
    geom_curve(data = winner_notes, aes(x = x, y = y, xend = xend, yend = yend),
               curvature = 0.3, color = '#333333', linewidth = 0.5,
               arrow = arrow(length = unit(0.15, 'cm'), type = 'closed'))

p6

## Step 7: title, subtitle and footer

`labs()` supplies the text; the theme styles it. `plot.title.position = 'plot'` left-aligns the title with the edge of the figure (the default aligns it with the panel), which is the FiveThirtyEight look. The caption doubles as the footer.

In [ ]:
p7 = p6 +
    labs(
        title    = "A slower field at this year's Boston Marathon",
        subtitle = 'Finish time for winners of the Boston Marathon, by country',
        caption  = 'Using unofficial times for 2018 winners.\n\nFiveThirtyEight\n\nSOURCE: BOSTON ATHLETIC ASSOCIATION'
    ) +
    theme(
        plot.title            = element_text(face = 'bold', size = 16),
        plot.subtitle         = element_text(size = 12, color = '#333333', margin = margin(b = 15)),
        plot.caption          = element_text(hjust = 0, size = 9, color = '#777777', margin = margin(t = 15)),
        plot.title.position   = 'plot',
        plot.caption.position = 'plot'
    )

p7